In [18]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import timm
from tqdm.notebook import tqdm
import numpy as np

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

In [19]:
import os
import shutil

for folder in [train_folder, test_folder]:
    cp_dir = os.path.join(folder, '.ipynb_checkpoints')
    if os.path.exists(cp_dir):
        shutil.rmtree(cp_dir)

In [20]:
class PlayingCardDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data = ImageFolder(data_dir, transform=transform)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]
    @property
    def classes(self):
        return self.data.classes

train_folder = './data/train'
test_folder = './data/test'

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

train_dataset = PlayingCardDataset(train_folder, transform=transform)
test_dataset = PlayingCardDataset(test_folder, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [21]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.base_model = timm.create_model('efficientnet_b0', pretrained=True)
        self.features = nn.Sequential(*list(self.base_model.children())[:-1])
        self.flatten = nn.Flatten()

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        return x

model = FeatureExtractor().to(device)
model.eval()

FeatureExtractor(
  (base_model): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv

In [22]:
train_features_list = []
train_labels_list = []

with torch.no_grad():
    for images, labels in tqdm(train_loader):
        images = images.to(device)
        feats = model(images)
        train_features_list.append(feats.cpu().numpy())
        train_labels_list.append(labels.numpy())

X_train = np.vstack(train_features_list)
y_train = np.concatenate(train_labels_list)

  0%|          | 0/239 [00:00<?, ?it/s]

In [23]:
test_features_list = []
test_labels_list = []

with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        feats = model(images)
        test_features_list.append(feats.cpu().numpy())
        test_labels_list.append(labels.numpy())

X_test = np.vstack(test_features_list)
y_test = np.concatenate(test_labels_list)

  0%|          | 0/9 [00:00<?, ?it/s]

In [24]:
K = 5
y_pred = []

similarity_matrix = cosine_similarity(X_test, X_train)

for similarities in tqdm(similarity_matrix):
    top_k_indices = np.argsort(similarities)[-K:][::-1]
    top_k_labels = y_train[top_k_indices]
    most_common_label = Counter(top_k_labels).most_common(1)[0][0]
    y_pred.append(most_common_label)

y_pred = np.array(y_pred)

  0%|          | 0/265 [00:00<?, ?it/s]

In [25]:
target_names = train_dataset.classes

print(classification_report(y_test, y_pred, target_names=target_names))
print(confusion_matrix(y_test, y_pred))

                   precision    recall  f1-score   support

     ace of clubs       0.40      0.40      0.40         5
  ace of diamonds       0.60      0.60      0.60         5
    ace of hearts       0.27      0.60      0.38         5
    ace of spades       0.62      1.00      0.77         5
   eight of clubs       0.67      0.40      0.50         5
eight of diamonds       0.33      0.20      0.25         5
  eight of hearts       0.43      0.60      0.50         5
  eight of spades       0.50      0.40      0.44         5
    five of clubs       0.12      0.20      0.15         5
 five of diamonds       0.27      0.60      0.38         5
   five of hearts       0.00      0.00      0.00         5
   five of spades       0.20      0.20      0.20         5
    four of clubs       0.00      0.00      0.00         5
 four of diamonds       0.67      0.40      0.50         5
   four of hearts       0.20      0.20      0.20         5
   four of spades       0.33      0.20      0.25       

C:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag